# GPU-2 public-evaluation comparison

This is a self-contained, inference-only development checkpoint. It evaluates one
separate smoke task and the frozen 16-task pilot selection from the public
evaluation challenges, then compares the model arm with the existing symbolic
fallback. It never writes a submission file and never loads evaluation solutions
until predictions and per-task artifacts are persisted.

Real mode requires the pinned Kaggle environment, four independent L4 workers,
the exact GPU-1 model/dependency attachments, and Internet disabled. Set
GPU2_MODE=cpu-check for the local synthetic orchestration check; this does not
claim GPU or model success. Set GPU2_OUTPUT_DIR to override the output path.

The GPU-1 definition, vendored payload, entry-adjacent checks, Apache-2.0 license
notice, and upstream attribution are retained in the hash-verified embedded
snapshot. GPU-2 calls those helpers for loading, reset, tokenization, formatting,
augmentation, DFS/ranking, validation, and supervision. The only new runtime
adapter selects inference_only; no TTT arm is invoked or reported.

The frozen selection is 581f7754 for smoke, followed by
16b78196,31f7f899,4c3d4a41,89565ca0,53fb4810,d35bdbdc,6e453dd6,7491f3cf,
446ef5d2,7b3084d4,1818057f,80a900e0,269e22fb,65b59efc,dfadab01,a251c730.
The checkpoint reports full-denominator metrics, completed-only metrics, missing,
error, timeout, fallback, timing, memory, and exact-grid added/lost IDs. This is
not clean unseen evaluation because the pretrained checkpoint may contain public
evaluation-derived material.


In [ ]:
from __future__ import annotations

import base64
import copy
import gzip
import hashlib
import json
import os
import signal
import sys
import time
import traceback
import types
from collections import Counter
from pathlib import Path
from typing import Any

GPU2_NOTEBOOK_SHA256 = (
    "463829abfea83fb95d7336dc6a4ffb8a7f652e3bcdb3947e840cf502bc7b70e4"
)
GPU2_SELECTION_SHA256 = (
    "5a90862bf29b78ae0f8d9a4a4c6eb66e9401b52b62f5e3c5ed67694d68bbbb56"
)
GPU2_SEED_LABEL = "gpu2-nvarc-public-evaluation-v1"
GPU2_SMOKE_COUNT = 1
GPU2_PILOT_COUNT = 16
GPU2_ARMS = ("inference_only",)
GPU2_GRACEFUL_SECONDS = 5100
GPU2_HARD_SECONDS = 5400
GPU2_CLEANUP_SECONDS = 30
GPU2_GPU_COUNT = 4
GPU2_MAX_SEQ_LENGTH = 8192
GPU2_INFERENCE_SECONDS = 1200
GPU2_OUTPUT_NAME = "gpu2-public-evaluation"
GPU2_EXPECTED_OUTPUTS = 22
GPU2_FROZEN_SELECTION = {
    "smoke": ["581f7754"],
    "pilot": [
        "16b78196",
        "31f7f899",
        "4c3d4a41",
        "89565ca0",
        "53fb4810",
        "d35bdbdc",
        "6e453dd6",
        "7491f3cf",
        "446ef5d2",
        "7b3084d4",
        "1818057f",
        "80a900e0",
        "269e22fb",
        "65b59efc",
        "dfadab01",
        "a251c730",
    ],
}
GPU2_REFERENCE_CELL_SHA256 = {
    "1": "f75d087e44de33e961e398207ce9f108402d9925ac1b6c5605cda12fd37b00e4",
    "2": "edc87e9d8247d70fc052698983df469abc417e9086af22724ed77b84437ccd6a",
    "3": "fdc0179b3031873234198e984fb9c4ed1fdb99bc8c6e993131a010458ffc722e",
}
GPU1_NOTEBOOK_GZIP_B64 = "..."  # noqa: E501


class GPU2Error(RuntimeError):
    pass


class MissingInputError(GPU2Error):
    pass


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _cell_source(cell: dict[str, Any]) -> str:
    source = cell["source"]
    return "".join(source) if isinstance(source, list) else source


def load_gpu1_reference() -> dict[str, Any]:
    notebook_bytes = gzip.decompress(base64.b64decode(GPU1_NOTEBOOK_GZIP_B64))
    if sha256_text(notebook_bytes.decode("utf-8")) != GPU2_NOTEBOOK_SHA256:
        raise GPU2Error("embedded GPU-1 notebook snapshot hash mismatch")
    notebook = json.loads(notebook_bytes)
    namespace: dict[str, Any] = {
        "__name__": "gpu2_gpu1_reference",
        "__file__": "gpu1_nvarc_pilot.ipynb",
    }
    for index in (1, 2, 3):
        source = _cell_source(notebook["cells"][index])
        if sha256_text(source) != GPU2_REFERENCE_CELL_SHA256[str(index)]:
            raise GPU2Error(f"embedded GPU-1 cell {index} hash mismatch")
        exec(
            compile(source, f"gpu1_nvarc_pilot.ipynb#cell-{index}", "exec"),
            namespace,
        )
    return namespace


REFERENCE = load_gpu1_reference()
# GPU-2 uses the trusted worker with one mutable namespace entry changed.
REFERENCE["arm_order"] = lambda: ("inference_only",)


def validate_inference_only_policy() -> tuple[str, ...]:
    order = REFERENCE["arm_order"]()
    REFERENCE["validate_condition_switch"](order)
    if order != GPU2_ARMS:
        raise GPU2Error("GPU-2 policy must contain only inference_only")
    return order


def _selection_ids(selection: dict[str, list[str]]) -> list[str]:
    return [*selection["smoke"], *selection["pilot"]]


def validate_public_selection(challenges: dict[str, Any]) -> dict[str, list[str]]:
    runtime_selection = REFERENCE["select_task_ids"](
        challenges,
        seed_label=GPU2_SEED_LABEL,
        smoke_count=GPU2_SMOKE_COUNT,
        pilot_count=GPU2_PILOT_COUNT,
    )
    if runtime_selection != GPU2_FROZEN_SELECTION:
        raise GPU2Error(
            f"frozen selection mismatch: expected {GPU2_FROZEN_SELECTION}, "
            f"got {runtime_selection}"
        )
    selected_ids = _selection_ids(runtime_selection)
    if len(selected_ids) != 17 or len(set(selected_ids)) != 17:
        raise GPU2Error("frozen selection must contain 17 unique task IDs")
    output_count = sum(
        len(challenges[task_id]["test"]) for task_id in runtime_selection["pilot"]
    )
    if output_count != GPU2_EXPECTED_OUTPUTS:
        raise GPU2Error(
            f"frozen selection output count changed: expected "
            f"{GPU2_EXPECTED_OUTPUTS}, got {output_count}"
        )
    return runtime_selection


def _discover_public_input(root: Path, filename: str, env_name: str) -> Path:
    try:
        return REFERENCE["_one_input"](root, filename, env_name)
    except REFERENCE["MissingInputError"] as exc:
        raise MissingInputError(str(exc)) from exc


def discover_evaluation_challenges(root: Path) -> Path:
    return _discover_public_input(
        root, "arc-agi_evaluation_challenges.json", "GPU2_EVALUATION_CHALLENGES"
    )


def discover_evaluation_solutions(root: Path) -> Path:
    return _discover_public_input(
        root, "arc-agi_evaluation_solutions.json", "GPU2_EVALUATION_SOLUTIONS"
    )


def _clock(started: float) -> Any:
    clock = REFERENCE["Deadline"](started)
    clock.graceful = started + GPU2_GRACEFUL_SECONDS
    clock.hard = started + GPU2_HARD_SECONDS - GPU2_CLEANUP_SECONDS
    return clock


def _first_attempt_predictions(
    predictions: dict[str, list[dict[str, Any]]],
) -> dict[str, list[dict[str, Any]]]:
    return {
        task_id: [
            {**prediction, "attempt_2": prediction["attempt_1"]}
            for prediction in task_predictions
        ]
        for task_id, task_predictions in predictions.items()
    }


def _score_arm(
    runtime: dict[str, Any],
    challenges: dict[str, Any],
    solutions: dict[str, Any],
    predictions: dict[str, list[dict[str, Any]]],
) -> dict[str, Any]:
    return {
        "either_attempt": runtime["score_submission"](
            challenges, solutions, predictions
        ),
        "first_attempt": runtime["score_submission"](
            challenges,
            solutions,
            _first_attempt_predictions(predictions),
        ),
    }


def _comparison_ids(
    challenges: dict[str, Any],
    solutions: dict[str, Any],
    results: dict[str, Any],
    task_ids: list[str],
    base: str,
    candidate: str,
) -> dict[str, dict[str, list[str]]]:
    output_ids: dict[str, dict[str, list[str]]] = {}
    task_ids_by_result: dict[str, dict[str, list[str]]] = {}
    for attempt_name, predicate in (
        (
            "either_attempt",
            lambda prediction, expected: (
                prediction["attempt_1"] == expected
                or prediction["attempt_2"] == expected
            ),
        ),
        (
            "first_attempt",
            lambda prediction, expected: prediction["attempt_1"] == expected,
        ),
    ):
        added_outputs: list[str] = []
        lost_outputs: list[str] = []
        added_tasks: list[str] = []
        lost_tasks: list[str] = []
        for task_id in task_ids:
            base_task = True
            candidate_task = True
            for index, (base_prediction, candidate_prediction, expected) in enumerate(
                zip(
                    results[task_id]["arms"][base]["predictions"],
                    results[task_id]["arms"][candidate]["predictions"],
                    solutions[task_id],
                    strict=True,
                )
            ):
                base_correct = predicate(base_prediction, expected)
                candidate_correct = predicate(candidate_prediction, expected)
                output_id = f"{task_id}:{index}"
                if candidate_correct and not base_correct:
                    added_outputs.append(output_id)
                if base_correct and not candidate_correct:
                    lost_outputs.append(output_id)
                base_task = base_task and base_correct
                candidate_task = candidate_task and candidate_correct
            if candidate_task and not base_task:
                added_tasks.append(task_id)
            if base_task and not candidate_task:
                lost_tasks.append(task_id)
        output_ids[attempt_name] = {
            "added_output_ids": added_outputs,
            "lost_output_ids": lost_outputs,
        }
        task_ids_by_result[attempt_name] = {
            "added_task_ids": added_tasks,
            "lost_task_ids": lost_tasks,
        }
    return {
        attempt_name: {
            **output_ids[attempt_name],
            **task_ids_by_result[attempt_name],
        }
        for attempt_name in output_ids
    }


def _first_attempt_results(
    results: dict[str, Any], task_ids: list[str], arms: tuple[str, ...]
) -> dict[str, Any]:
    adjusted = {}
    for task_id in task_ids:
        adjusted[task_id] = {**results[task_id], "arms": {}}
        for arm in arms:
            entry = results[task_id]["arms"][arm]
            adjusted[task_id]["arms"][arm] = {
                **entry,
                "predictions": _first_attempt_predictions(
                    {task_id: entry["predictions"]}
                )[task_id],
            }
    return adjusted


def _comparison_delta(
    challenges: dict[str, Any],
    solutions: dict[str, Any],
    results: dict[str, Any],
    task_ids: list[str],
    base: str,
    candidate: str,
) -> dict[str, Any]:
    either_counts = REFERENCE["_comparison_delta"](
        challenges, solutions, results, base, candidate, task_ids
    )
    first_results = _first_attempt_results(results, task_ids, (base, candidate))
    first_counts = REFERENCE["_comparison_delta"](
        challenges, solutions, first_results, base, candidate, task_ids
    )
    ids = _comparison_ids(challenges, solutions, results, task_ids, base, candidate)
    return {
        "either_attempt": {
            **either_counts,
            **ids["either_attempt"],
        },
        "first_attempt": {
            **first_counts,
            **ids["first_attempt"],
        },
    }


def _coverage(results: dict[str, Any]) -> dict[str, Any]:
    entries = [
        result["arms"]["inference_only"]
        for result in results.values()
        if "inference_only" in result.get("arms", {})
    ]
    status_counts = Counter(entry["status"] for entry in entries)
    return {
        "tasks": len(results),
        "arms": {
            "inference_only": {
                "status": dict(status_counts),
                "missing": sum(entry["status"] == "missing" for entry in entries),
                "error": sum(entry["status"] == "error" for entry in entries),
                "timeout": sum(entry["status"] == "timeout" for entry in entries),
                "fallback_outputs": sum(
                    entry.get("diagnostics", {}).get("fallback_outputs", 0)
                    for entry in entries
                ),
                "model_outputs": sum(
                    entry.get("diagnostics", {}).get("model_outputs", 0)
                    for entry in entries
                ),
            }
        },
    }


def score_results(
    runtime: dict[str, Any],
    challenges: dict[str, Any],
    solutions: dict[str, Any],
    results: dict[str, Any],
) -> dict[str, Any]:
    task_ids = list(results)
    for task_id in task_ids:
        REFERENCE["_fill_result"](
            runtime, task_id, challenges[task_id], results[task_id]
        )
    arms = ("symbolic", *validate_inference_only_policy())
    predictions = {
        arm: {
            task_id: results[task_id]["arms"][arm]["predictions"]
            for task_id in task_ids
        }
        for arm in arms
    }
    full_metrics = {
        arm: _score_arm(runtime, challenges, solutions, predictions[arm])
        for arm in arms
    }
    completed_ids = [
        task_id
        for task_id in task_ids
        if results[task_id].get("worker_errors") is None
        and all(
            results[task_id]["arms"].get(arm, {}).get("status") == "complete"
            and not any(
                results[task_id]["arms"]
                .get(arm, {})
                .get("diagnostics", {})
                .get("fallback_mask", [])
            )
            for arm in validate_inference_only_policy()
        )
    ]
    completed_challenges = {task_id: challenges[task_id] for task_id in completed_ids}
    completed_solutions = {task_id: solutions[task_id] for task_id in completed_ids}
    completed_metrics = (
        {
            arm: _score_arm(
                runtime,
                completed_challenges,
                completed_solutions,
                {task_id: predictions[arm][task_id] for task_id in completed_ids},
            )
            for arm in arms
        }
        if completed_ids
        else {}
    )
    comparisons = {
        "inference_only_vs_symbolic": _comparison_delta(
            challenges, solutions, results, task_ids, "symbolic", "inference_only"
        )
    }
    error = any(result.get("worker_errors") for result in results.values())
    error = error or any(
        entry["status"] == "error"
        for result in results.values()
        for entry in result["arms"].values()
    )
    fallback = any(
        any(
            result["arms"]["inference_only"]
            .get("diagnostics", {})
            .get("fallback_mask", [])
        )
        for result in results.values()
    )
    status = (
        "error"
        if error
        else "incomplete"
        if fallback or len(completed_ids) != len(task_ids)
        else "complete"
    )
    return {
        "status": status,
        "denominator": {
            "tasks": len(task_ids),
            "outputs": sum(len(challenges[task_id]["test"]) for task_id in task_ids),
        },
        "metrics": {
            "full_frozen": full_metrics,
            "completed_only": {"task_ids": completed_ids, **completed_metrics},
        },
        "comparisons": comparisons,
        "coverage": _coverage(results),
        "errors": {
            task_id: result["worker_errors"]
            for task_id, result in results.items()
            if result.get("worker_errors")
        },
        "by_task": {
            task_id: {
                "worker_errors": result.get("worker_errors", []),
                **{
                    arm: {
                        "status": result["arms"][arm]["status"],
                        "timing": result["arms"][arm].get("timing", {}),
                        "memory": result["arms"][arm].get("memory", {}),
                        "diagnostics": result["arms"][arm].get("diagnostics", {}),
                    }
                    for arm in arms
                },
            }
            for task_id, result in results.items()
        },
    }


def _cpu_modules() -> dict[str, Any]:
    class CPUArcDataset:
        def __init__(self, queries: dict[str, Any]) -> None:
            self.queries = queries
            self.keys = list(queries)

        @classmethod
        def from_file(cls, path: str, keys: list[str]) -> CPUArcDataset:
            challenges = REFERENCE["load_json"](Path(path))
            return cls({task_id: challenges[task_id] for task_id in keys})

        def change_keys(self, _keys: list[str]) -> CPUArcDataset:
            return self

    class CPUModel:
        def __init__(self) -> None:
            self.adapter_state = {"value": 0}
            self.inference_only = False

    class CPUFastLanguageModel:
        @staticmethod
        def for_inference(model: CPUModel) -> CPUModel:
            model.inference_only = True
            return model

    class CPUTorch:
        class CUDA:
            @staticmethod
            def reset_peak_memory_stats() -> None:
                return None

            @staticmethod
            def max_memory_allocated() -> int:
                return 0

    solver = types.SimpleNamespace(
        FastLanguageModel=CPUFastLanguageModel,
        torch=types.SimpleNamespace(cuda=CPUTorch.CUDA()),
        set_peft_model_state_dict=lambda model, value, adapter_name: setattr(
            model, "adapter_state", copy.deepcopy(value)
        ),
        get_peft_model_state_dict=lambda model, adapter_name: copy.deepcopy(
            model.adapter_state
        ),
    )
    return {
        "arc_loader": types.SimpleNamespace(ArcDataset=CPUArcDataset),
        "arc_solver": solver,
    }


def _cpu_decode(
    _model: Any,
    task_dataset: Any,
    _tokenizer: Any,
    _modules: dict[str, Any],
    _end_time: float,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    task_id = task_dataset.keys[0]
    return [
        {
            "attempt_1": copy.deepcopy(test["input"]),
            "attempt_2": copy.deepcopy(test["input"]),
        }
        for test in task_dataset.queries[task_id]["test"]
    ], {"status": "complete", "synthetic_model": True}


def cpu_inference_worker(
    _rank: int,
    task_ids: list[str],
    result_queue: Any,
    config: dict[str, Any],
    started: float,
) -> None:
    try:
        modules = _cpu_modules()
        # The child uses the reference worker with a cheap decoder substitution.
        REFERENCE["_decode_arm"] = _cpu_decode
        for task_id in task_ids:
            task_result = REFERENCE["run_gpu_task"](
                types.SimpleNamespace(),
                types.SimpleNamespace(),
                modules,
                {"value": 0},
                Path(config["training_path"]),
                task_id,
                started + 60,
            )
            result_queue.put({"kind": "task", **task_result})
    except Exception as exc:
        result_queue.put(
            {
                "kind": "worker_error",
                "task_ids": task_ids,
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(),
            }
        )


def run_cpu_check(output_dir: Path) -> dict[str, Any]:
    if output_dir.exists() and any(output_dir.iterdir()):
        raise GPU2Error(f"refusing to overwrite existing run output: {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=False)
    evaluation_path = output_dir / "synthetic-evaluation-challenges.json"
    REFERENCE["write_json_atomic"](
        evaluation_path,
        {
            "00000001": {
                "train": [{"input": [[0]], "output": [[1]]}],
                "test": [{"input": [[2]]}, {"input": [[3]]}],
            }
        },
    )
    clock = _clock(time.monotonic())
    supervised = REFERENCE["supervise_workers"](
        [["00000001"]],
        cpu_inference_worker,
        {"training_path": str(evaluation_path)},
        clock,
    )
    task_events = [
        event for event in supervised["events"] if event.get("kind") == "task"
    ]
    if len(task_events) != 1 or task_events[0]["status"] != "complete":
        raise GPU2Error(f"synthetic inference orchestration failed: {supervised}")
    arm_result = task_events[0]["arms"]["inference_only"]
    if (
        not arm_result["training_succeeded"]
        or not arm_result["diagnostics"]["adapter_reset_verified"]
    ):
        raise GPU2Error("CPU substitution did not exercise inference-only adapter")
    result = {
        "mode": "cpu-check",
        "status": "passed",
        "orchestration": {
            "events": len(supervised["events"]),
            "task_ids": ["00000001"],
            "arms": list(validate_inference_only_policy()),
            "training_called": False,
            "adapter_reset_verified": arm_result["diagnostics"][
                "adapter_reset_verified"
            ],
            "model_outputs": len(arm_result["predictions"]),
        },
        "reference": {
            "notebook_sha256": GPU2_NOTEBOOK_SHA256,
            "selection_sha256": GPU2_SELECTION_SHA256,
            "cell_sha256": GPU2_REFERENCE_CELL_SHA256,
        },
        "limitations": [
            "synthetic worker and model only",
            "CUDA, model loading, inference accuracy, memory, and timing unverified",
        ],
    }
    REFERENCE["write_json_atomic"](output_dir / "cpu-check.json", result)
    return result


def _fingerprint(path: Path) -> dict[str, Any]:
    return {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": REFERENCE["sha256_file"](path),
    }


def _write_manifest(path: Path, manifest: dict[str, Any]) -> None:
    REFERENCE["write_json_atomic"](path / "manifest.json", manifest)


def run_real(output_dir: Path) -> dict[str, Any]:
    REFERENCE["check_python_abi"]()
    if output_dir.exists() and any(output_dir.iterdir()):
        raise GPU2Error(f"refusing to overwrite existing run output: {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / "tasks").mkdir()
    started = time.monotonic()
    clock = _clock(started)
    symbolic_runtime = REFERENCE["load_symbolic_runtime"]()
    input_root = Path(os.environ.get("GPU2_INPUT_ROOT", "/kaggle/input"))
    setup_started = time.monotonic()
    challenges_path = discover_evaluation_challenges(input_root)
    challenges = REFERENCE["load_json"](challenges_path)
    symbolic_runtime["validate_challenges"](challenges)
    selection = validate_public_selection(challenges)
    selected_ids = _selection_ids(selection)
    scored_ids = selection["pilot"]
    model_dir = REFERENCE["discover_model_dir"](input_root)
    dependency_root = REFERENCE["discover_dependency_root"](input_root)
    manifest = {
        "schema_version": 1,
        "experiment": GPU2_OUTPUT_NAME,
        "mode": "real",
        "started_epoch": time.time(),
        "reference": {
            "notebook_sha256": GPU2_NOTEBOOK_SHA256,
            "definition_payload_entry_cell_sha256": GPU2_REFERENCE_CELL_SHA256,
            "upstream_version": REFERENCE["UPSTREAM_VERSION"],
            "upstream_source_sha256": REFERENCE["UPSTREAM_SOURCE_SHA256"],
            "offline_setup_version": REFERENCE["OFFLINE_SETUP_VERSION"],
            "offline_setup_source_sha256": REFERENCE["OFFLINE_SETUP_SOURCE_SHA256"],
            "vendored_source_sha256": REFERENCE["VENDORED_SOURCE_SHA256"],
            "symbolic_runtime_sha256": REFERENCE["runtime_source_fingerprint"](),
        },
        "configuration": {
            "seed_label": GPU2_SEED_LABEL,
            "selection": selection,
            "arms": list(validate_inference_only_policy()),
            "max_seq_length": GPU2_MAX_SEQ_LENGTH,
            "graceful_seconds": GPU2_GRACEFUL_SECONDS,
            "hard_seconds": GPU2_HARD_SECONDS,
            "cleanup_reserve_seconds": GPU2_CLEANUP_SECONDS,
            "gpu_workers": GPU2_GPU_COUNT,
            "inference": {
                "augmentation_n": 2,
                "seed": 2,
                "ranking": "score_kgmon",
                "seconds_per_arm": GPU2_INFERENCE_SECONDS,
            },
        },
        "inputs": {
            "evaluation_challenges": _fingerprint(challenges_path),
            "model": REFERENCE["model_fingerprint"](model_dir),
            "dependencies": REFERENCE["dependency_fingerprint"](dependency_root),
        },
        "runtime": {"python": sys.version, "platform": sys.platform},
        "answers_loaded": False,
        "stage": "setup",
        "timing": {},
    }
    manifest["configuration_sha256"] = sha256_text(
        json.dumps(manifest["configuration"], sort_keys=True)
    )
    manifest["timing"]["setup_seconds"] = time.monotonic() - setup_started
    _write_manifest(output_dir, manifest)
    print("GPU-2 setup complete", flush=True)
    config = {
        "model_dir": str(model_dir),
        "dependency_root": str(dependency_root),
        # The trusted worker names this input training_path; the schema is identical.
        "training_path": str(challenges_path),
    }
    preflight_started = time.monotonic()
    print("GPU-2 preflight started", flush=True)
    preflight = REFERENCE["supervise_workers"](
        [[]], REFERENCE["gpu_preflight_worker"], config, clock
    )
    preflight_events = [
        event for event in preflight["events"] if event.get("kind") == "preflight"
    ]
    if not preflight_events or preflight_events[-1].get("status") != "complete":
        manifest["preflight"] = preflight_events[-1] if preflight_events else preflight
        manifest["stage"] = "preflight_error"
        _write_manifest(output_dir, manifest)
        raise GPU2Error(f"GPU preflight failed: {manifest['preflight']}")
    manifest["preflight"] = preflight_events[-1]
    manifest["timing"]["preflight_seconds"] = time.monotonic() - preflight_started
    manifest["stage"] = "preflight_complete"
    _write_manifest(output_dir, manifest)
    print("GPU-2 preflight complete", flush=True)

    results: dict[str, dict[str, Any]] = {
        task_id: {
            "task_id": task_id,
            "status": "model_missing",
            "task": challenges[task_id],
            "arms": {},
        }
        for task_id in selected_ids
    }

    def save_event(event: dict[str, Any]) -> None:
        print(
            "GPU-2 event:",
            {
                key: event[key]
                for key in ("kind", "task_id", "arm", "status", "rank")
                if key in event
            },
            flush=True,
        )
        REFERENCE["append_event"](output_dir / "events.jsonl", event)
        event_task_ids = (
            [event["task_id"]]
            if event.get("task_id") is not None
            else event.get("task_ids", [])
        )
        for task_id in event_task_ids:
            if task_id not in results:
                continue
            if event.get("kind") == "arm":
                merge_event = {
                    "task_id": task_id,
                    "arms": {event["arm"]: event["result"]},
                }
            elif event.get("kind") == "worker_error":
                merge_event = {
                    "task_id": task_id,
                    "status": "error",
                    "error": event.get("error"),
                    "traceback": event.get("traceback"),
                }
                results[task_id].setdefault("worker_errors", []).append(
                    {
                        "error": event.get("error"),
                        "traceback": event.get("traceback"),
                    }
                )
                REFERENCE["append_event"](
                    output_dir / "events.jsonl",
                    {**merge_event, "kind": "task", "source": "worker_error"},
                )
            else:
                merge_event = event
            REFERENCE["_merge_task_event"](results, merge_event)
            REFERENCE["write_json_atomic"](
                output_dir / "tasks" / f"{task_id}.json", results[task_id]
            )

    smoke_started = time.monotonic()
    smoke_ids = selection["smoke"]
    print("GPU-2 smoke started", flush=True)
    smoke = REFERENCE["supervise_workers"](
        [smoke_ids], REFERENCE["gpu_worker"], config, clock, save_event
    )
    smoke_entry = results[smoke_ids[0]]
    smoke_arm = smoke_entry.get("arms", {}).get("inference_only", {})
    smoke_diagnostics = smoke_arm.get("diagnostics", {})
    smoke_no_candidate = (
        smoke_entry.get("status") == "complete"
        and not smoke_entry.get("worker_errors")
        and smoke_arm.get("status") == "complete"
        and smoke_arm.get("model_loaded") is True
        and smoke_arm.get("training_succeeded") is True
        and smoke_diagnostics.get("adapter_reset_verified") is True
        and smoke_diagnostics.get("candidate_count") == 0
        and smoke_diagnostics.get("invalid_candidates") == 0
        and smoke_diagnostics.get("fallback_outputs")
        == len(challenges[smoke_ids[0]]["test"])
        and smoke_arm.get("predictions")
        == [{} for _ in challenges[smoke_ids[0]]["test"]]
    )
    smoke_valid = (
        REFERENCE["_smoke_is_valid"](
            symbolic_runtime, smoke_ids[0], challenges[smoke_ids[0]], smoke_entry
        )
        or smoke_no_candidate
    )
    if (
        not smoke_valid
        or smoke_entry.get("status") != "complete"
        or smoke_entry.get("worker_errors")
    ):
        manifest["smoke"] = smoke_entry
        manifest["stage"] = "smoke_error"
        _write_manifest(output_dir, manifest)
        raise GPU2Error(f"smoke task failed validation: {smoke_entry}")
    manifest["smoke"] = {
        "task_id": smoke_ids[0],
        "status": "complete",
        "no_candidate": smoke_no_candidate,
    }
    if smoke_no_candidate:
        manifest["smoke"].update(
            {
                "zero_coverage": True,
                "coverage": {
                    "candidate_count": smoke_diagnostics["candidate_count"],
                    "invalid_candidates": smoke_diagnostics["invalid_candidates"],
                    "fallback_outputs": smoke_diagnostics["fallback_outputs"],
                },
            }
        )
    manifest["timing"]["smoke_seconds"] = time.monotonic() - smoke_started
    manifest["stage"] = "smoke_complete"
    _write_manifest(output_dir, manifest)
    print("GPU-2 smoke complete", flush=True)

    pilot_started = time.monotonic()
    print("GPU-2 pilot started", flush=True)
    pilot = REFERENCE["supervise_workers"](
        [scored_ids[rank::GPU2_GPU_COUNT] for rank in range(GPU2_GPU_COUNT)],
        REFERENCE["gpu_worker"],
        config,
        clock,
        save_event,
    )
    for task_id in selected_ids:
        REFERENCE["_fill_result"](
            symbolic_runtime, task_id, challenges[task_id], results[task_id]
        )
        REFERENCE["write_json_atomic"](
            output_dir / "tasks" / f"{task_id}.json", results[task_id]
        )
    predictions_path = output_dir / "predictions.json"
    REFERENCE["write_json_atomic"](
        predictions_path,
        {
            task_id: {
                arm: results[task_id]["arms"][arm]["predictions"]
                for arm in ("symbolic", "inference_only")
            }
            for task_id in selected_ids
        },
    )
    manifest["timing"]["pilot_seconds"] = time.monotonic() - pilot_started
    manifest["timing"]["predictions_persisted_seconds"] = time.monotonic() - started
    manifest["stage"] = "predictions_persisted"
    _write_manifest(output_dir, manifest)
    print("GPU-2 predictions persisted", flush=True)

    report_started = time.monotonic()
    solutions_path = discover_evaluation_solutions(input_root)
    solutions = REFERENCE["load_json"](solutions_path)
    selected_challenges = {task_id: challenges[task_id] for task_id in scored_ids}
    selected_solutions = {task_id: solutions[task_id] for task_id in scored_ids}
    symbolic_runtime["validate_solutions"](selected_challenges, selected_solutions)
    manifest["inputs"]["evaluation_solutions"] = _fingerprint(solutions_path)
    manifest["answers_loaded"] = True
    manifest["answers_loaded_stage"] = "after_predictions_persisted"
    manifest["stage"] = "answers_loaded"
    _write_manifest(output_dir, manifest)
    summary = score_results(
        symbolic_runtime,
        selected_challenges,
        selected_solutions,
        {task_id: results[task_id] for task_id in scored_ids},
    )
    if pilot["hard_timeout"] or smoke["hard_timeout"]:
        summary["status"] = "deadline_partial"
    manifest["timing"]["report_seconds"] = time.monotonic() - report_started
    manifest["status"] = summary["status"]
    manifest["stage"] = "complete"
    manifest["timing"]["total_seconds"] = time.monotonic() - started
    _write_manifest(output_dir, manifest)
    summary["timing"] = manifest["timing"]
    summary["label"] = (
        "GPU-2 public-evaluation development checkpoint; "
        "pretrained checkpoint may include public-evaluation-derived material"
    )
    REFERENCE["write_json_atomic"](output_dir / "summary.json", summary)
    print("GPU-2 report complete", flush=True)
    return summary


def notebook_output_dir(mode: str) -> Path:
    configured = os.environ.get("GPU2_OUTPUT_DIR")
    if configured:
        return Path(configured)
    if mode == "real" and Path("/kaggle/working").is_dir():
        return Path("/kaggle/working") / GPU2_OUTPUT_NAME
    return Path("artifacts/gpu2-public-evaluation")


def main() -> dict[str, Any]:
    mode = os.environ.get("GPU2_MODE", "real").strip().lower()
    if mode not in {"real", "cpu-check"}:
        raise GPU2Error("GPU2_MODE must be real or cpu-check")
    if mode == "cpu-check":
        return run_cpu_check(notebook_output_dir(mode))

    def hard_stop(_signum: int, _frame: Any) -> None:
        raise GPU2Error("global hard deadline reached, including synchronous setup")

    watchdog_started = time.monotonic()
    previous_handler = signal.signal(signal.SIGALRM, hard_stop)
    previous_timer = signal.setitimer(signal.ITIMER_REAL, GPU2_HARD_SECONDS)
    try:
        return run_real(notebook_output_dir(mode))
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, previous_handler)
        if previous_timer[0] > 0:
            signal.setitimer(
                signal.ITIMER_REAL,
                max(
                    0.001,
                    previous_timer[0] - (time.monotonic() - watchdog_started),
                ),
                previous_timer[1],
            )

In [ ]:
result = main()
print(json.dumps(result, indent=2, sort_keys=True)[:16000])